# Peta Perubahan Tutupan Lahan Kabupaten Banjar 2025 vs 2026 — Versi Google Earth Engine (GEE)

**Mata kuliah:** Sistem Informasi Geografi (SIG) — **Dosen:** Dr. Ir. Mufidah Asy'ari, M.P.
**Wilayah:** Kabupaten Banjar, Kalimantan Selatan (luas BPS 4.668,50 km²)
**Citra:** Landsat 8 + Landsat 9, Collection 2 Level-2 Surface Reflectance, komposit median Juni–September 2025 vs 2026
**Metode:** Random Forest supervised (`ee.Classifier.smileRandomForest`, 300 pohon), dilatih **pooled** pada komposit 2025 DAN 2026
**Fitur (14):** band B2–B7 + NDVI, NDWI, NDBI, EVI, NDMI, BSI + elevasi + kemiringan (SRTM 30 m)
**Kelas (5):** 1 Hutan · 2 Pertanian & Perkebunan · 3 Permukiman / Lahan Terbangun · 4 Badan Air & Rawa · 5 Lahan Terbuka & Semak Belukar

> **Versi ini memakai Google Earth Engine.** Ada juga versi tanpa-login yang sama persis metodenya memakai Microsoft Planetary Computer: `landcover_change_banjar_2025_2026_PC.ipynb` — pakai versi itu kalau belum punya akses GEE.

> **Catatan hasil:** versi Planetary Computer (metode sama) mencapai OA 2025 = 0,679, Kappa = 0,599; OA 2026 = 0,689, Kappa = 0,611 — di bawah target OA ≥ 0,85. Notebook GEE ini **belum pernah dieksekusi end-to-end** (belum ada login GEE di mesin pembuatnya); jalankan sendiri di Colab dengan akun GEE Anda, lalu bandingkan hasilnya dengan versi PC. Lihat README untuk penjelasan lengkap.

## 0. Prasyarat — baca dulu (sekali saja)

1. **Daftar Earth Engine:** buka https://code.earthengine.google.com , login dengan akun Google, klik **Sign up** / terima ketentuannya. Akun baru biasanya langsung aktif.
2. **Buat Google Cloud project:** buka https://console.cloud.google.com , buat project baru (misal `gee-banjar`), lalu di project itu aktifkan **Earth Engine API** (menu APIs & Services → Enable APIs → cari "Earth Engine API").
3. **Isi `PROJECT_ID`** pada cell konfigurasi di bawah dengan nama project dari langkah 2.
4. Saat cell autentikasi dijalankan, Colab menampilkan link — buka link itu, pilih akun Google yang sudah terdaftar di Earth Engine, salin kode/tokennya, lalu **paste ke kotak yang muncul di Colab**.

Kalau muncul error `EEException: ... not registered` berarti langkah 1 belum selesai; kalau error `project not found` berarti `PROJECT_ID` salah.

## 1. Install package

In [ ]:
!pip install -q earthengine-api geemap
print("install selesai — lanjut ke cell berikutnya")

## 2. Autentikasi GEE

In [28]:
import ee

# Jalankan cell ini, lalu paste token autentikasi saat Colab memintanya.
ee.Authenticate()

# GANTI dengan nama Google Cloud project Anda (dari langkah prasyarat 2).
PROJECT_ID = "whatsapp-drive-bot-512115"

ee.Initialize(project=PROJECT_ID)
print("GEE aktif. Project:", PROJECT_ID)

GEE aktif. Project: whatsapp-drive-bot-512115


## 3. Konfigurasi dan batas wilayah

In [29]:
import geemap

TAHUN_A, TAHUN_B = 2025, 2026
N_POHON = 300
N_TITIK_PER_KELAS = 2500
SEED = 42
SKALA_M = 30
CRS = "EPSG:32650"  # UTM zona 50S

KELAS = {
    1: "Hutan",
    2: "Pertanian & Perkebunan",
    3: "Permukiman / Lahan Terbangun",
    4: "Badan Air & Rawa",
    5: "Lahan Terbuka & Semak Belukar",
}
WC_DARI = [10, 95, 40, 50, 80, 90, 20, 30, 60]
WC_KE   = [1, 1, 2, 3, 4, 4, 5, 5, 5]
BAND_SR = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]
BAND_INDEKS = ["NDVI", "NDWI", "NDBI", "EVI", "NDMI", "BSI"]
BAND_DEM = ["elevasi", "kemiringan"]
BAND_FITUR = BAND_SR + BAND_INDEKS + BAND_DEM   # 14 fitur

# Batas Kabupaten Banjar dari geoBoundaries ADM2 (sumber BPS).
# 'Banjar' yang polos hanya ada satu di ADM2 Indonesia (yang di Kalsel);
# filterBounds dipakai sebagai pengaman agar tidak tertukar dengan
# Banjarnegara / Kota Banjar.
roi = (ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM2")
       .filter(ee.Filter.eq("shapeName", "Banjar"))
       .filterBounds(ee.Geometry.Rectangle([114.5, -3.74, 115.6, -2.82]))
       .geometry())
print("Luas batas Banjar (geoBoundaries):",
      round(roi.area().divide(1e6).getInfo(), 2), "km2 (BPS: 4.668,50 km2)")

Luas batas Banjar (geoBoundaries): 4743.37 km2 (BPS: 4.668,50 km2)


## 4. Komposit median Landsat Jun–Sep 2025 dan 2026

In [30]:
def mask_awan(img):
    """Mask awan/bayangan/cirrus dari QA_PIXEL (bit 1,2,3,4)."""
    qa = img.select("QA_PIXEL")
    mask = (qa.bitwiseAnd(1 << 1).eq(0)
            .And(qa.bitwiseAnd(1 << 2).eq(0))
            .And(qa.bitwiseAnd(1 << 3).eq(0))
            .And(qa.bitwiseAnd(1 << 4).eq(0)))
    return img.updateMask(mask)

def skala_sr(img):
    """SR (DN) -> reflektansi: 0,0000275 * DN - 0,2."""
    opt = img.select(BAND_SR).multiply(0.0000275).add(-0.2)
    return img.addBands(opt, None, True)

def tambah_indeks(img):
    ndvi = img.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
    ndwi = img.normalizedDifference(["SR_B3", "SR_B5"]).rename("NDWI")
    ndbi = img.normalizedDifference(["SR_B6", "SR_B5"]).rename("NDBI")
    evi = img.expression(
        "2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)",
        {"NIR": img.select("SR_B5"), "RED": img.select("SR_B4"),
         "BLUE": img.select("SR_B2")}).rename("EVI")
    ndmi = img.normalizedDifference(["SR_B5", "SR_B6"]).rename("NDMI")
    bsi = img.expression(
        "((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))",
        {"SWIR1": img.select("SR_B6"), "RED": img.select("SR_B4"),
         "NIR": img.select("SR_B5"), "BLUE": img.select("SR_B2")}).rename("BSI")
    return img.addBands([ndvi, ndwi, ndbi, evi, ndmi, bsi])

def tambah_dem(img):
    """Elevasi SRTM 30 m (negatif -> 0) + kemiringan."""
    srtm = ee.Image("USGS/SRTMGL1_003").clip(roi)
    elev = srtm.rename("elevasi").max(0)
    slope = ee.Terrain.slope(srtm).rename("kemiringan")
    return img.addBands([elev, slope])

def komposit(tahun):
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
           .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
           .filterBounds(roi)
           .filterDate(f"{tahun}-06-01", f"{tahun}-09-30")
           .map(mask_awan).map(skala_sr).map(tambah_indeks)
           .select(BAND_SR + BAND_INDEKS))
    print(f"Komposit {tahun}: {col.size().getInfo()} scene dipakai")
    return tambah_dem(col.median().clip(roi))

komp_2025 = komposit(2025)
komp_2026 = komposit(2026)

Komposit 2025: 41 scene dipakai
Komposit 2026: 45 scene dipakai


## 5. Training sample dari ESA WorldCover 2021

> **Jujur soal data:** tidak ada survei lapangan dalam tenggat tugas ini, jadi label latih/validasi diambil dari ESA WorldCover 2021 yang dipetakan ke 5 kelas. Artinya akurasi yang dihitung adalah akurasi **terhadap referensi WorldCover 2021**, bukan akurasi lapangan, dan perubahan lahan 2021→2025/2026 bisa menimbulkan bias. Hal ini juga ditulis di README dan laporan.

In [31]:
# WorldCover 2021 -> 5 kelas
wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
kelas_img = wc.remap(WC_DARI, WC_KE, 0).rename("kelas")

# Kemurnian: jumlah piksel sekelas di tetangga 3x3 (0-9); sampel hanya
# yang >= 8. Padanan di GEE untuk kemurnian WorldCover 10 m versi PC.
def peta_kemurnian(kelas_img):
    total = ee.Image(0)
    for c in KELAS:
        satu = kelas_img.eq(c)
        cnt = satu.reduceNeighborhood(
            reducer=ee.Reducer.sum(),
            kernel=ee.Kernel.square(1, "pixels", False))  # normalize=False agar hasilnya hitungan 0-9, bukan rata-rata 0-1
        total = total.where(kelas_img.eq(c), cnt)
    return total.rename("kemurnian")
kemurnian = peta_kemurnian(kelas_img)

# Sampel stratified (2500 titik per kelas), hanya piksel murni >=8/9,
# split 70% latih / 30% validasi
sampel = (komp_2025.addBands(kelas_img).addBands(kemurnian)
          .stratifiedSample(
              numPoints=N_TITIK_PER_KELAS, classBand="kelas", region=roi,
              scale=SKALA_M, seed=SEED, geometries=True))
sampel = sampel.filter(ee.Filter.gte("kemurnian", 8))
sampel = sampel.randomColumn("acak", SEED)
latih = sampel.filter(ee.Filter.lt("acak", 0.7))
validasi = sampel.filter(ee.Filter.gte("acak", 0.7))
print("Sampel latih:", latih.size().getInfo())
print("Sampel validasi:", validasi.size().getInfo())

Sampel latih: 4967
Sampel validasi: 2113


## 6. Klasifikasi Random Forest (pooled)

Model **pooled**: lokasi latih yang sama dipakai dua kali — sekali dengan fitur 2025, sekali dengan fitur 2026 — sehingga satu model stabil untuk kedua tahun.

In [32]:
# POOLED: lokasi latih yang sama dipakai dengan fitur 2026 juga
latih_2026 = komp_2026.sampleRegions(
    collection=latih, scale=SKALA_M, geometries=False)
latih_pooled = latih.merge(latih_2026)
print("Baris latih pooled:", latih_pooled.size().getInfo())
clf = ee.Classifier.smileRandomForest(N_POHON).train(
    latih_pooled, "kelas", BAND_FITUR)
kls_2025 = komp_2025.classify(clf).rename("kelas").toUint8()
kls_2026 = komp_2026.classify(clf).rename("kelas").toUint8()
print("Klasifikasi 2025 dan 2026 selesai.")

Baris latih pooled: 9934
Klasifikasi 2025 dan 2026 selesai.


## 7. Uji akurasi per tahun (validasi 30%)

In [33]:
def akurasi(validasi_fc, nama):
    pred_val = validasi_fc.classify(clf)
    matriks = pred_val.errorMatrix("kelas", "classification")
    print(f"--- Akurasi {nama} (validasi 30%, label WorldCover 2021) ---")
    print("Overall Accuracy:", round(matriks.accuracy().getInfo(), 4))
    print("Kappa:", round(matriks.kappa().getInfo(), 4))
    print("Producer's accuracy per kelas:",
          [round(x, 3) for x in matriks.producersAccuracy().getInfo()])
    print("User's accuracy per kelas:",
          [round(x, 3) for x in matriks.consumersAccuracy().getInfo()])
    print("Confusion matrix (baris=referensi, kolom=prediksi):")
    print(matriks.array().getInfo())

# validasi sudah membawa fitur 2025; untuk 2026 ambil fitur di titik sama
validasi_2026 = komp_2026.sampleRegions(
    collection=validasi, scale=SKALA_M, geometries=False)
akurasi(validasi, "validasi 2025 (model pooled)")
akurasi(validasi_2026, "validasi 2026 (model pooled)")
# Target tugas: OA >= 0,85 dan Kappa >= 0,80. Versi PC (metode sama)
# mencapai OA 0,679 / 0,689 — di bawah target; laporkan apa adanya dan
# jelaskan penyebabnya di README/laporan (label WorldCover 2021 sudah
# tua, tumpang tindih spektral, musim kemarau 2026 lebih kering).

--- Akurasi validasi 2025 (model pooled) (validasi 30%, label WorldCover 2021) ---
Overall Accuracy: 0.8254
Kappa: 0.7725


TypeError: type list doesn't define __round__ method

## 8. Luas per kelas dan matriks transisi

In [40]:
def hitung_luas(img_kelas, nama):
    luas = (ee.Image.pixelArea().addBands(img_kelas)
            .reduceRegion(reducer=ee.Reducer.sum().group(
                groupField=1, groupName="kelas"),
                geometry=roi, scale=SKALA_M, maxPixels=1e13))
    total = 0.0
    print(f"--- Luas {nama} ---")
    for g in luas.get("groups").getInfo():
        k, ha = int(g["kelas"]), g["sum"] / 1e4
        total += ha
        print(f"  {k} {KELAS[k]:35s} {ha:12,.1f} ha")
    print(f"  TOTAL {total:,.1f} ha")
    return total

hitung_luas(kls_2025, "klasifikasi 2025")
hitung_luas(kls_2026, "klasifikasi 2026")

# Matriks transisi 2025 -> 2026 (ha); kode perubahan = kelas2025*10 + kelas2026
kode = kls_2025.multiply(10).add(kls_2026).rename("perubahan")
out = (ee.Image.pixelArea().addBands(kode)
       .reduceRegion(reducer=ee.Reducer.sum().group(
           groupField=1, groupName="perubahan"),
           geometry=roi, scale=SKALA_M, maxPixels=1e13))
mat = {(a, b): 0.0 for a in KELAS for b in KELAS}
for g in out.get("groups").getInfo():
    v = int(g["perubahan"])
    a, b = v // 10, v % 10
    if (a, b) in mat:
        mat[(a, b)] = g["sum"] / 1e4
print("\nMatriks transisi 2025 -> 2026 (ha):")
for a in KELAS:
    print(f"{KELAS[a][:30]:30s} " + " ".join(f"{mat[(a, b)]:10,.0f}" for b in KELAS))
trans = sorted(((v, a, b) for (a, b), v in mat.items() if a != b), reverse=True)[:3]
print("\n3 transisi perubahan terbesar:")
for v, a, b in trans:
    print(f"  {KELAS[a]} -> {KELAS[b]}: {v:,.1f} ha")

--- Luas klasifikasi 2025 ---
  1 Hutan                                  314,085.1 ha
  2 Pertanian & Perkebunan                  61,793.6 ha
  3 Permukiman / Lahan Terbangun             7,150.3 ha
  4 Badan Air & Rawa                        20,646.6 ha
  5 Lahan Terbuka & Semak Belukar           62,906.9 ha
  TOTAL 466,582.5 ha
--- Luas klasifikasi 2026 ---
  1 Hutan                                  281,842.6 ha
  2 Pertanian & Perkebunan                  58,641.2 ha
  3 Permukiman / Lahan Terbangun            16,392.3 ha
  4 Badan Air & Rawa                        22,706.9 ha
  5 Lahan Terbuka & Semak Belukar           91,963.2 ha
  TOTAL 471,546.3 ha

Matriks transisi 2025 -> 2026 (ha):
Hutan                             273,057      2,889      1,918        386     35,804
Pertanian & Perkebunan                233     42,154      3,877      6,848      8,681
Permukiman / Lahan Terbangun           74        424      5,964         69        620
Badan Air & Rawa                      231  

## 9. Tampilkan peta

In [42]:
palet = ["#1a9850", "#fee08b", "#d73027", "#2166ac", "#bdbdbd"]
# Palet perubahan: kode 11,22,33,44,55 (tidak berubah) = abu-abu muda,
# kode perubahan lain = warna tegas (urutan kode 11..55).
palet_ubah = ["#f0f0f0" if (i // 5) == (i % 5) else c for i, c in enumerate(
    ["#a6cee3", "#1f78b4", "#b2df8a", "#33a02c", "#fb9a99"] * 5)]
Map = geemap.Map(center=[-3.32, 115.05], zoom=10)
Map.add_basemap("HYBRID")
Map.addLayer(kls_2025, {"min": 1, "max": 5, "palette": palet},
             "Klasifikasi 2025")
Map.addLayer(kls_2026, {"min": 1, "max": 5, "palette": palet},
             "Klasifikasi 2026")
Map.addLayer(kode, {"min": 11, "max": 55, "palette": palet_ubah},
             "Perubahan 2025-2026 (kode = 2025*10+2026)", False)
Map.add_legend(title="Tutupan lahan",
               legend_dict={KELAS[k]: palet[k-1] for k in KELAS})
Map

Map(center=[-3.32, 115.05], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## 10. Ekspor hasil ke Google Drive

In [43]:
for img, nama in [(kls_2025, "klasifikasi_2025"),
                      (kls_2026, "klasifikasi_2026"),
                      (kode.toUint8(), "perubahan_2025_2026")]:
    tugas = ee.batch.Export.image.toDrive(
        image=img, description=nama, folder="GEE_Banjar",
        fileNamePrefix=nama, region=roi, scale=SKALA_M,
        crs=CRS, maxPixels=1e13)
    tugas.start()
    print("Ekspor dimulai:", nama, "-> Google Drive / GEE_Banjar /")
print("Cek progres di https://code.earthengine.google.com/tasks — tunggu status 'Completed', lalu file GeoTIFF ada di Drive Anda.")

Ekspor dimulai: klasifikasi_2025 -> Google Drive / GEE_Banjar /
Ekspor dimulai: klasifikasi_2026 -> Google Drive / GEE_Banjar /
Ekspor dimulai: perubahan_2025_2026 -> Google Drive / GEE_Banjar /
Cek progres di https://code.earthengine.google.com/tasks — tunggu status 'Completed', lalu file GeoTIFF ada di Drive Anda.


## 11. Unduh hasil (opsional)

File GeoTIFF hasil ekspor tersimpan di Google Drive (folder `GEE_Banjar`). Untuk mengunduhnya ke Colab / komputer:
- Buka https://drive.google.com, masuk ke folder `GEE_Banjar`, unduh ketiga file GeoTIFF.
- Atau mount Drive di Colab (`from google.colab import drive; drive.mount('/content/drive')`) lalu salin filenya.

## Catatan akhir
- Akurasi di cell 7 adalah akurasi terhadap label **WorldCover 2021**, bukan cek lapangan.
- Bila OA < 0,85 atau Kappa < 0,80, tambahkan titik sampel (naikkan `N_TITIK_PER_KELAS`), lalu ulangi cell 5–7; bila dua kelas saling tertukar parah, gabungkan kelas tersebut dan laporkan alasannya.
- Versi tanpa-login dengan metode yang sama: `landcover_change_banjar_2025_2026_PC.ipynb` (Planetary Computer).